## 09C_06 - Benchmark Colosseo (caso reale: chatbot per i visitatori; tag e classi di distanza curati dal gestore del sito)
Stesso schema di `09_04_ArmstrongBenchmark` (Armstrong resta l'esperimento "in vitro"); **runner e configurazioni (C0-C5) restano quelli di `benchmark_runner.py`**.

**File** in `~/tesi_graphrag/data/queries/colosseum/`: `chunks_colosseum_final.json`, `queries_predetermined.json`, `queries_normal.json`, `eval_queries.json`, `gold_chunks.json`.
**Moduli** in `src/evaluation/`: `json_ingest.py`, `distance_presets.py`, `colosseum_curation.py`, `degrade_tags.py` (+ patch `check_similarity` con `paired_with` opzionale).

**Dataset**: 120 chunk (Wikipedia + articolo Cerone + pagina sui sotterranei), 6 tag, 42 query (14 ambigue, 14 specialistiche, 14 multi-dominio), **12 predeterminate (~30%) scelte a caso con seed**.
Le ambigue hanno una formulazione con piu' letture: il tag di conversazione (`expected_tag`) dice quale e' voluta, i tag e le distanze dei chunk devono aiutare a sceglierla.

**Due condizioni** (niente `pre_edit`, e' ridondante), stessi tag di conversazione e stesse query:
- `post_edit_expert`: **tag e distanze ottimali** (sidecar `colosseum_benchmark.json`) = tetto del sistema
- `post_edit_user`: **tag con errori/mancanti** (`degrade_tags`) **e distanze parziali** (`degrade_plan`) (sidecar `colosseum_benchmark_user.json`) = utente medio

C0 (puro vettoriale) non dipende ne' da tag ne' da distanze: si valuta una volta sola, **nel run `post_edit_expert`**. Le altre config usano tag/predq/chunk<->chunk anche senza `DIST`, quindi vanno valutate in entrambi i run.

## Ordine di esecuzione 09C_06

**Prima di partire:** `COLLECTION_NAME = "colosseum_bench"` in `src/config.py`, riavvio del kernel.
JSON del dataset in `data/queries/colosseum/`; moduli `json_ingest.py`, `distance_presets.py`, `colosseum_curation.py`, `degrade_tags.py` in `src/evaluation/`.

1. **Cella 0** (init): percorsi e parametri (`TOP_J=5`, `TOP_K=15`, `TOP_N=7`). Una volta per kernel.
2. **Cella 1** (ingestione): collezione Qdrant + tag dei chunk nel sidecar. **Una volta sola.**
3. **Cella 2** (handler + benchmark): crea `handler`, `bench`, `sm` e controlla il dataset. Da rilanciare se cambi i JSON delle query.
4. **Cella 3** (predq + similarita'): registra le 12 predeterminate (**una volta per sidecar**) e lancia `check_similarity`.
   Normali con similarita' >= 0.80: riformulale nel JSON, rilancia la cella 2 e il solo `check_similarity`.
5. **Cella 4**: stampa conv_tag e manual_tags per controllo. **Cella 5**: backup del sidecar "base" (tag + predq, nessuna distanza).
6. **PILOTA A** (nessun LLM, ~1 min): le ambigue sono davvero ambigue? Quante query hanno margine di miglioramento rispetto a C0?
7. *(opzionale)* **PILOTA B** (LLM, ~15 min su CPU): Llama sa gia' rispondere senza i chunk? Segnala le domande "troppo note".
8. **Versione esperta**: genera il piano da `colosseum_curation.py`, **rivedilo**, applicalo. **PILOTA C** (nessun LLM): il reranking migliora C0 sulle query *normali*?
   Solo se il verdetto e' favorevole, genera `post_edit_expert` (con risposte).
9. **Versione utente medio**: tag degradati + piano ridotto, copia del sidecar base, secondo handler. **PILOTA D** (nessun LLM), poi `post_edit_user`.
10. **RAGAS su Colab** (`RagasColab_T4_v3.ipynb`, `DATASET = "colosseum"`): `post_edit_expert` (6 config, C0 inclusa) e `post_edit_user` (5 config). Scarica i CSV in `data/benchmark/colosseum/`.
11. **Report**: per categoria, in-sample contro normali, differenze appaiate con C0, retrieval sui gold, **verdetto per il relatore** (con e senza risposta umana).

**Attenzioni**
- La generazione e' riprendibile; `generate()` salta cio' che e' gia' salvato: dopo aver cambiato parametri, query o piano usa un run_id nuovo (i pilota hanno run_id propri e non toccano i run veri).
- Un'unica istanza di `SidecarManager` per file (`sm = handler.sidecar_manager`).
- Il piano esperto e' un tetto: chunk, query e taratura sono scritti insieme. Se ritocchi la curation guardando i risultati sulle query normali, dichiaralo in tesi (rischio di adattamento al test).
- Non cambiare `TOP_J/TOP_K/TOP_N` tra i due run.

In [1]:
### 0 ### Inizializzazione - una volta per kernel
import sys, json, shutil, logging
from pathlib import Path
import pandas as pd

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.sidecar_manager import SidecarManager
from src.evaluation.benchmark_runner import RagasBenchmark
from src.evaluation import distance_presets as dp
from src.evaluation import colosseum_curation as cu
from src.evaluation.json_ingest import ingest_chunks_json, load_tags_into_sidecar
from src.evaluation.degrade_tags import degrade_tags

#logging.basicConfig(level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", datefmt="%H:%M:%S", force=True)

COLLECTION = "colosseum_bench"
QDIR = project_root / "data/queries/colosseum"
sidecar_path = project_root / "data/processed/colosseum/colosseum_benchmark.json"
base_sidecar = sidecar_path.with_name("colosseum_benchmark_base.json")     # tag + predq, nessuna distanza
chunks_path = QDIR / "chunks_colosseum_final.json"
queries_path = QDIR / "eval_queries.json"
pred_path, norm_path, gold_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json", QDIR / "gold_chunks.json"
OUT_DIR = project_root / "data/benchmark/colosseum"
TOP_J, TOP_K, TOP_N = 5, 15, 7   # ~120 chunk: top_k=15 e' circa il 12% del corpus
META = ["run_id", "query_id", "category", "subtype", "is_predetermined", "config"]
GOLD_COLS = ["gold_hit", "gold_recall", "gold_precision", "gold_mrr"]

### 1 - Ingestione (UNA volta). Ricrea la collezione e scrive i tag dei chunk nel sidecar.
Prima di lanciarla: in `src/config.py` imposta `COLLECTION_NAME = "colosseum_bench"` e riavvia il kernel (l'handler legge la collezione da li').

In [ ]:
sidecar_path.parent.mkdir(parents=True, exist_ok=True)
ingest_chunks_json(chunks_path, COLLECTION, sidecar_path).release_path()   # NB: azzera i tag del sidecar e li riscrive; distanze e predq restano

In [2]:
### 2 ### Handler + benchmark (lanciare dopo l'ingestione, con COLLECTION_NAME in config.py)
handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)
sm = handler.sidecar_manager      # UNICA istanza del sidecar: usare sempre questa
bench = RagasBenchmark(handler, predetermined_path=pred_path, normal_path=norm_path,
                       out_dir=OUT_DIR,
                       top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)

# Controlli sul dataset
allq = json.load(open(queries_path, encoding="utf-8")); gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
assert len({q["id"] for q in allq}) == len(allq) == 42
assert sum(q["is_predetermined"] for q in allq) == 12
assert all(g in chunks for v in gold.values() for g in v), "gold con chunk inesistenti"
tags = {t for c in chunks.values() for t in c["tags"]}
TAGS = sorted(tags)
assert all(q["expected_tag"] in tags | {None} and set(q["tags"]) <= tags for q in allq), "tag di query non presenti sui chunk"
print(len(chunks), "chunk;", len(allq), "query")
print(pd.DataFrame(allq).groupby(["category", "is_predetermined"]).size().unstack())
print("tag nel sidecar:", {k: v["count"] for k, v in sm.get_global_tags().items()})

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


120 chunk; 42 query
is_predetermined  False  True 
category                      
ambiguous            10      4
multi_domain         10      4
specialistic         10      4
tag nel sidecar: {'History': 48, 'Architecture': 48, 'Culture': 19, 'Spectacles': 28, 'Conservation': 14, 'Archaeology': 29}


In [3]:
### 3 ### Registrazione delle 12 predeterminate (UNA volta per sidecar) + controllo similarita'
registered = bench.register_predetermined("pred")
print(len(registered), "predq registrate")
sim = bench.check_similarity(warn_above=0.80)    # richiede la patch con paired_with opzionale
display(sim.head(8))     # le normali >= 0.80 vanno riformulate (la soglia dell'handler e' 0.85)

12 predq registrate


!> 1 query normali troppo simili a una predeterminata: ['MUL_01']


,normal_id,max_sim,closest_pred
20,MUL_01,0.826329,MUL_09
14,SPE_08,0.783214,AMB_05
25,MUL_07,0.777352,AMB_05
9,AMB_14,0.775193,AMB_05
19,SPE_13,0.761778,MUL_12
21,MUL_03,0.752021,SPE_02
22,MUL_04,0.751463,MUL_11
10,SPE_01,0.749940,SPE_02


In [4]:
### 4 ### Query: conv_tag e manual_tags che il runner usera' (dal JSON)
display(pd.DataFrame(allq)[["id", "category", "subtype", "expected_tag", "tags", "is_predetermined"]])

,id,category,subtype,expected_tag,tags,is_predetermined
0,AMB_01,ambiguous,modern_vs_ancient,Conservation,[],True
1,AMB_02,ambiguous,build_vs_use,Archaeology,[],True
2,AMB_03,ambiguous,evidence_vs_accounts,Archaeology,[],False
3,AMB_04,ambiguous,statue_vs_name,History,[],False
4,AMB_05,ambiguous,name_meaning,Culture,[],True
5,AMB_06,ambiguous,damage_cause,Conservation,[],False
6,AMB_07,ambiguous,paintings,Culture,[],False
7,AMB_08,ambiguous,living_quarters,Architecture,[],False
8,AMB_09,ambiguous,site_today,Archaeology,[],False
9,AMB_10,ambiguous,which_restoration,Conservation,[],False


In [5]:
### 5 ### Backup del sidecar "base": tag + predq registrate, NESSUNA distanza (da qui parte anche l'utente medio)
shutil.copy(sidecar_path, base_sidecar)
print("backup:", base_sidecar.name)

backup: colosseum_benchmark_base.json


### PILOTA A - le ambigue sono davvero ambigue? C'e' margine di miglioramento? (nessun LLM, ~1 minuto)
Esegue solo il **retrieval** (`with_answers=False`) di tutte le query su C0-C5 con i soli tag (nessuna distanza ancora).
- **C0 per categoria**: se le ambigue hanno gold_recall/MRR piu' bassi delle specialistiche, l'ambiguita' c'e'.
- **Margine**: le query in cui C0 non recupera tutto il gold nei top-J sono quelle dove il reranking puo' migliorare. Se sono troppo poche
  (es. < 8 su 42) il dataset non puo' dimostrare nulla: servono query piu' ambigue.
- **C1 contro C0**: i soli tag, senza distanze, migliorano gia' le ambigue?

In [6]:
### PILOTA A ###
genA = bench.generate("pilot_tags_only", with_answers=False)
mA = genA[META + GOLD_COLS]
c0 = genA[genA.config == "C0_baseline"].set_index("query_id")
print("C0 - gold per categoria (piu' basso = piu' difficile per il solo vettoriale)")
display(c0.groupby("category")[GOLD_COLS].mean().round(2))
headroom = c0[c0.gold_recall < 1]
print(f"\nQuery con margine (C0 non recupera tutto il gold nei top-{TOP_J}): {len(headroom)}/{len(c0)}")
display(headroom.groupby("category").size().rename("n_con_margine").to_frame())
display(headroom[["category", "subtype", "is_predetermined", "gold_hit", "gold_recall", "gold_mrr"]].sort_values("gold_recall"))
print("\nC1 (tag) contro C0, tutte le query:")
display(bench.paired_vs_baseline(mA[mA.config.isin(["C0_baseline", "C1_tags"])]))
print("solo ambigue:")
display(bench.paired_vs_baseline(mA[mA.config.isin(["C0_baseline", "C1_tags"])], subset=(mA.category == "ambiguous")))

C0 - gold per categoria (piu' basso = piu' difficile per il solo vettoriale)


,gold_hit,gold_recall,gold_precision,gold_mrr
category,,,,
ambiguous,0.71,0.35,0.20,0.60
multi_domain,0.79,0.45,0.34,0.58
specialistic,0.79,0.73,0.26,0.64



Query con margine (C0 non recupera tutto il gold nei top-5): 32/42


,n_con_margine
category,
ambiguous,14
multi_domain,13
specialistic,5


,category,subtype,is_predetermined,gold_hit,gold_recall,gold_mrr
query_id,,,,,,
AMB_01,ambiguous,modern_vs_ancient,True,0,0.000000,0.000000
SPE_02,specialistic,crowd_flow,True,0,0.000000,0.000000
AMB_08,ambiguous,living_quarters,False,0,0.000000,0.000000
MUL_12,multi_domain,lost_and_found,True,0,0.000000,0.000000
SPE_08,specialistic,statue_move,False,0,0.000000,0.000000
SPE_13,specialistic,sphinx_room,False,0,0.000000,0.000000
AMB_13,ambiguous,sailors,False,0,0.000000,0.000000
AMB_12,ambiguous,ships,False,0,0.000000,0.000000
MUL_07,multi_domain,nero_to_flavian,False,0,0.000000,0.000000



C1 (tag) contro C0, tutte le query:


,metric,config,n,n_nan,mean_diff,ci95_low,ci95_high,win,tie,loss
0,gold_hit,C1_tags,42,0,0.071,-0.024,0.167,4,37,1
1,gold_recall,C1_tags,42,0,0.068,0.014,0.128,11,27,4
2,gold_precision,C1_tags,42,0,0.043,0.005,0.086,11,27,4
3,gold_mrr,C1_tags,42,0,0.066,-0.019,0.162,8,31,3


solo ambigue:


/home/jovyan/tesi_graphrag/src/evaluation/benchmark_runner.py:365: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  d = df if subset is None else df[subset]


,metric,config,n,n_nan,mean_diff,ci95_low,ci95_high,win,tie,loss
0,gold_hit,C1_tags,14,0,0.214,0.000,0.429,3,11,0
1,gold_recall,C1_tags,14,0,0.167,0.071,0.286,6,8,0
2,gold_precision,C1_tags,14,0,0.100,0.043,0.171,6,8,0
3,gold_mrr,C1_tags,14,0,0.214,0.036,0.429,4,10,0


### PILOTA B (opzionale) - Llama sa gia' rispondere senza i chunk? (LLM, ~15 minuti su CPU)
Colosseo e' molto noto: se Llama risponde bene anche **senza contesto**, la domanda misura poco il retrieval.
Per ogni query: risposta senza chunk e risposta con i chunk di C0; similarita' coseno (embedding locali) di ciascuna con la risposta di riferimento.
Le query con `sim_senza_contesto >= sim_con_C0 - 0.03` sono "troppo note": se sono molte, riformulale su dettagli presenti solo nei chunk.
(Proxy economico, senza giudice: serve per decidere, non va riportato come metrica.)

In [7]:
### PILOTA B ###
import numpy as np
NOCTX = OUT_DIR / "pilot_noctx_answers.jsonl"
done = {json.loads(l)["query_id"] for l in open(NOCTX, encoding="utf-8")} if NOCTX.exists() else set()
with open(NOCTX, "a", encoding="utf-8") as f:
    for q in allq:
        if q["id"] in done: continue
        out = handler.llm.invoke(f"Answer the question concisely (2-3 sentences) using only what you know.\nQuestion: {q['query']}\nAnswer:")
        f.write(json.dumps({"query_id": q["id"], "answer": (out.content if hasattr(out, "content") else str(out)).strip()}, ensure_ascii=False) + "\n"); f.flush()
noctx = pd.read_json(NOCTX, lines=True).set_index("query_id")

all_cfgs = bench.configs
bench.configs = [c for c in all_cfgs if c["name"] == "C0_baseline"]
genB = bench.generate("pilot_c0_answers").set_index("query_id")
bench.configs = all_cfgs

E = handler.embeddings
def _sim(a, b):
    va, vb = np.array(E.embed_query(a)), np.array(E.embed_query(b))
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))
rows = [{"query_id": q["id"], "category": q["category"],
         "sim_senza_contesto": _sim(noctx.loc[q["id"], "answer"], q["reference"]),
         "sim_con_C0": _sim(genB.loc[q["id"], "response"], q["reference"])} for q in allq]
dfB = pd.DataFrame(rows).set_index("query_id")
dfB["troppo_nota"] = dfB.sim_senza_contesto >= dfB.sim_con_C0 - 0.03
print("medie per categoria:"); display(dfB.groupby("category")[["sim_senza_contesto", "sim_con_C0"]].mean().round(3))
print(f"query 'troppo note': {int(dfB.troppo_nota.sum())}/{len(dfB)}"); display(dfB[dfB.troppo_nota].round(3))

medie per categoria:


,sim_senza_contesto,sim_con_C0
category,,
ambiguous,0.457,0.557
multi_domain,0.551,0.618
specialistic,0.523,0.671


query 'troppo note': 17/42


,category,sim_senza_contesto,sim_con_C0,troppo_nota
query_id,,,,
AMB_04,ambiguous,0.741,0.715,True
AMB_05,ambiguous,0.852,0.833,True
AMB_06,ambiguous,0.585,0.520,True
AMB_07,ambiguous,0.338,0.288,True
AMB_08,ambiguous,0.492,0.492,True
AMB_13,ambiguous,0.169,0.136,True
SPE_01,specialistic,0.833,0.781,True
SPE_02,specialistic,0.781,0.571,True
SPE_04,specialistic,0.800,0.791,True


In [10]:
dfB["delta_C0_vs_noctx"] = dfB["sim_con_C0"] - dfB["sim_senza_contesto"]
dfB["distanza_dalla_soglia"] = dfB["delta_C0_vs_noctx"] - 0.03

dfB_sorted = dfB.sort_values(by="delta_C0_vs_noctx", ascending=True)

with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.width", 1000
):
    print(f"=== TABELLA COMPLETA 42 QUERY (PILOTA B) ===")
    print(
        f"Query 'troppo note' (delta <= 0.03): {int(dfB.troppo_nota.sum())}/{len(dfB)}\n"
    )
    display(
        dfB_sorted[
            [
                "category",
                "sim_senza_contesto",
                "sim_con_C0",
                "delta_C0_vs_noctx",
                "distanza_dalla_soglia",
                "troppo_nota",
            ]
        ].round(3)
    )

=== TABELLA COMPLETA 42 QUERY (PILOTA B) ===
Query 'troppo note' (delta <= 0.03): 17/42



,category,sim_senza_contesto,sim_con_C0,delta_C0_vs_noctx,distanza_dalla_soglia,troppo_nota
query_id,,,,,,
SPE_13,specialistic,0.487,0.166,-0.321,-0.351,True
SPE_02,specialistic,0.781,0.571,-0.209,-0.239,True
MUL_12,multi_domain,0.574,0.433,-0.141,-0.171,True
MUL_08,multi_domain,0.548,0.409,-0.139,-0.169,True
MUL_03,multi_domain,0.773,0.664,-0.109,-0.139,True
SPE_09,specialistic,0.753,0.684,-0.069,-0.099,True
AMB_06,ambiguous,0.585,0.520,-0.065,-0.095,True
SPE_01,specialistic,0.833,0.781,-0.052,-0.082,True
AMB_07,ambiguous,0.338,0.288,-0.050,-0.080,True


In [11]:
out_path = Path(OUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

dfB_out = dfB.copy().reset_index()
dfB_out["delta_C0_vs_noctx"] = (
    dfB_out["sim_con_C0"] - dfB_out["sim_senza_contesto"]
)
dfB_out["distanza_dalla_soglia"] = dfB_out["delta_C0_vs_noctx"] - 0.03

csv_file = out_path / "pilot_b_results.csv"
dfB_out.to_csv(csv_file, index=False)

print(f"Risultati salvati in: {csv_file.resolve()}")

Risultati salvati in: /home/jovyan/tesi_graphrag/data/benchmark/colosseum/pilot_b_results.csv


### 6 - Versione ESPERTA (tetto): tag e distanze ottimali
I **tag** sono quelli rivisti a mano (`colosseum_review.json`) e gia' nel sidecar. Le **distanze**: `colosseum_curation.py` contiene, scritti a mano leggendo i chunk,
i gruppi di chunk che raccontano lo stesso episodio o tema (nucleo = classe forte, altri membri = lieve), i gruppi trasversali ai documenti
e per ogni predq i chunk di supporto (avvicinati) e quelli della **lettura sbagliata** dell'ambiguita' (allontanati).
In piu', con `bench`, vengono allontanati i chunk sbagliati che il retrieval vettoriale mette davvero nei top_j.
Il piano e' un JSON leggibile: **generalo, aprilo, ritoccalo (cancella/aggiungi coppie), poi applicalo**.

In [12]:
plan_expert = dp.build_expert_plan(allq, gold, sm, cu, bench=bench)   # bench=None -> niente penalizzazioni da retrieval; include_cross=False -> solo gruppi interni
plan_expert_path = QDIR / "distance_plan_colosseum_expert.json"
dp.save_plan(plan_expert, plan_expert_path)
print(plan_expert["classes"]); print(dp.plan_stats(plan_expert)); print("-> ritocca se serve:", plan_expert_path)

{'strong_up': 'MOLTO_AVVICINATI', 'mild_up': 'AVVICINATI', 'strong_down': 'MOLTO_ALLONTANATI', 'mild_down': 'ALLONTANATI'}
{'predq_edits': 99, 'chunk_pairs': 499, 'classes': {'MOLTO_AVVICINATI': 175, 'AVVICINATI': 380, 'MOLTO_ALLONTANATI': 19, 'ALLONTANATI': 24}}
-> ritocca se serve: /home/jovyan/tesi_graphrag/data/queries/colosseum/distance_plan_colosseum_expert.json


In [13]:
plan_expert = dp.load_plan(plan_expert_path)         # rilegge l'eventuale versione ritoccata a mano
print(dp.apply_plan(sm, plan_expert))                # azzera le adiacenze esistenti e scrive il piano
display(bench.edit_status().query("n_dist_edits > 0").head(20))

<<| Batch adiacenze query pre-determinate salvato: 99 elementi processati |>>
<<| Batch adiacenze standard salvato: 499 elementi processati |>>
{'predq_edits': 99, 'chunk_pairs': 499}


,query_id,registered,n_dist_edits,n_graph_tags
0,AMB_01,True,10,1
1,AMB_02,True,8,1
2,AMB_05,True,9,1
3,AMB_11,True,9,1
4,SPE_02,True,8,1
5,SPE_03,True,6,1
6,SPE_04,True,7,1
7,SPE_14,True,6,1
8,MUL_02,True,10,1
9,MUL_09,True,7,1


### PILOTA C - il reranking migliora C0 sulle query **normali**? (nessun LLM, ~1 minuto)
Dopo aver applicato il piano esperto: retrieval di tutte le query su tutte le config e confronto **appaiato con C0** sui gold.
Le predq (12) sono "in-sample" (le hai ritoccate); le **normali** misurano la generalizzazione. Verdetto: una config "migliora" se l'IC95% della differenza
di `gold_recall` o `gold_mrr` sulle normali e' tutto > 0 (almeno `win > loss`). Se **nessuna** config migliora C0 su normali e ambigue, e' inutile lanciare RAGAS:
guarda `distance_plan_colosseum_expert.json`, le ambigue con margine (PILOTA A) e rivedi tag/gruppi in `colosseum_review.json` / `colosseum_curation.py`.
Run id nuovo a ogni modifica del piano (es. `pilot_expert_v2`).

In [14]:
### PILOTA C ###
PC = "pilot_expert"
genC = bench.generate(PC, with_answers=False)
mC = genC[META + GOLD_COLS]
normal = ~mC.is_predetermined
def verdetto(m, subset, label):
    r = bench.paired_vs_baseline(m, subset=subset)
    r = r[r.metric.isin(["gold_recall", "gold_mrr"])].copy()
    r["migliora"] = (r.ci95_low > 0)
    r["peggiora"] = (r.ci95_high < 0)
    print(f"--- {label}"); display(r[["metric", "config", "n", "mean_diff", "ci95_low", "ci95_high", "win", "tie", "loss", "migliora", "peggiora"]])
    return r
rN = verdetto(mC, normal, "query NORMALI (generalizzazione)")
rA = verdetto(mC, normal & (mC.category == "ambiguous"), "AMBIGUE normali")
rP = verdetto(mC, ~normal, "predq (in-sample, solo per controllo: devono migliorare)")
print("config che migliorano C0 sulle normali:", sorted(set(rN[rN.migliora].config)) or "NESSUNA")

--- query NORMALI (generalizzazione)


,metric,config,n,mean_diff,ci95_low,ci95_high,win,tie,loss,migliora,peggiora
5,gold_recall,C1_tags,30,0.125,0.028,0.233,11,16,3,True,False
6,gold_recall,C2_dist,30,0.111,0.022,0.217,8,21,1,True,False
7,gold_recall,C3_dist_tags,30,0.103,0.006,0.211,9,18,3,True,False
8,gold_recall,C4_dist_tags_joinconv,30,0.106,0.011,0.211,10,17,3,True,False
9,gold_recall,C5_dist_tags_joinall,30,0.100,0.025,0.197,7,22,1,True,False
15,gold_mrr,C1_tags,30,0.051,-0.056,0.161,7,20,3,False,False
16,gold_mrr,C2_dist,30,0.032,-0.066,0.136,5,22,3,False,False
17,gold_mrr,C3_dist_tags,30,0.036,-0.069,0.144,5,22,3,False,False
18,gold_mrr,C4_dist_tags_joinconv,30,0.056,-0.053,0.167,7,20,3,False,False
19,gold_mrr,C5_dist_tags_joinall,30,0.078,-0.008,0.178,6,21,3,False,False


--- AMBIGUE normali


,metric,config,n,mean_diff,ci95_low,ci95_high,win,tie,loss,migliora,peggiora
5,gold_recall,C1_tags,10,0.333,0.167,0.533,7,3,0,True,False
6,gold_recall,C2_dist,10,0.300,0.133,0.500,6,4,0,True,False
7,gold_recall,C3_dist_tags,10,0.300,0.133,0.500,6,4,0,True,False
8,gold_recall,C4_dist_tags_joinconv,10,0.333,0.167,0.533,7,3,0,True,False
9,gold_recall,C5_dist_tags_joinall,10,0.267,0.067,0.467,5,5,0,True,False
15,gold_mrr,C1_tags,10,0.225,0.050,0.450,4,6,0,True,False
16,gold_mrr,C2_dist,10,0.200,0.050,0.400,3,7,0,True,False
17,gold_mrr,C3_dist_tags,10,0.200,0.050,0.400,3,7,0,True,False
18,gold_mrr,C4_dist_tags_joinconv,10,0.225,0.050,0.450,4,6,0,True,False
19,gold_mrr,C5_dist_tags_joinall,10,0.200,0.050,0.400,3,7,0,True,False


--- predq (in-sample, solo per controllo: devono migliorare)


,metric,config,n,mean_diff,ci95_low,ci95_high,win,tie,loss,migliora,peggiora
5,gold_recall,C1_tags,12,0.218,0.069,0.403,6,6,0,True,False
6,gold_recall,C2_dist,12,0.218,0.069,0.403,6,6,0,True,False
7,gold_recall,C3_dist_tags,12,0.218,0.069,0.403,6,6,0,True,False
8,gold_recall,C4_dist_tags_joinconv,12,0.322,0.158,0.503,8,4,0,True,False
9,gold_recall,C5_dist_tags_joinall,12,0.471,0.311,0.625,10,2,0,True,False
15,gold_mrr,C1_tags,12,0.306,0.097,0.542,5,7,0,True,False
16,gold_mrr,C2_dist,12,0.306,0.097,0.542,5,7,0,True,False
17,gold_mrr,C3_dist_tags,12,0.306,0.097,0.542,5,7,0,True,False
18,gold_mrr,C4_dist_tags_joinconv,12,0.306,0.097,0.542,5,7,0,True,False
19,gold_mrr,C5_dist_tags_joinall,12,0.306,0.097,0.542,5,7,0,True,False


config che migliorano C0 sulle normali: ['C1_tags', 'C2_dist', 'C3_dist_tags', 'C4_dist_tags_joinconv', 'C5_dist_tags_joinall']


In [15]:
### Dettaglio per query (esperto): dove C4 cambia rispetto a C0 sulle normali
piv = mC.pivot_table(index="query_id", columns="config", values="gold_recall")
d = (piv["C4_dist_tags_joinconv"] - piv["C0_baseline"]).rename("delta_recall_C4_vs_C0")
det = pd.concat([piv[["C0_baseline", "C4_dist_tags_joinconv"]], d], axis=1).join(pd.DataFrame(allq).set_index("id")[["category", "expected_tag", "is_predetermined"]])
display(det[det.delta_recall_C4_vs_C0 != 0].sort_values("delta_recall_C4_vs_C0"))

,C0_baseline,C4_dist_tags_joinconv,delta_recall_C4_vs_C0,category,expected_tag,is_predetermined
query_id,,,,,,
MUL_06,0.750000,0.250000,-0.500000,multi_domain,Spectacles,False
MUL_05,0.750000,0.500000,-0.250000,multi_domain,Conservation,False
MUL_10,0.250000,0.000000,-0.250000,multi_domain,Spectacles,False
MUL_02,0.600000,0.800000,0.200000,multi_domain,Spectacles,True
MUL_04,0.500000,0.750000,0.250000,multi_domain,Culture,False
MUL_01,0.500000,0.750000,0.250000,multi_domain,History,False
AMB_08,0.000000,0.333333,0.333333,ambiguous,Architecture,False
AMB_06,0.333333,0.666667,0.333333,ambiguous,Conservation,False
MUL_08,0.000000,0.333333,0.333333,multi_domain,Spectacles,False


In [20]:
gen = bench.generate("post_edit_expert")      # con risposte (LLM): lanciarla solo se il PILOTA C e' favorevole
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
display(bench.retrieval_summary("post_edit_expert"))

predq non riconosciute/errate: <ArrowStringArray>
[]
Length: 0, dtype: str


gold_hit  gold_recall  gold_precision  \
config                category                                              
C0_baseline           ambiguous        0.714        0.345           0.200   
                      multi_domain     0.786        0.448           0.343   
                      specialistic     0.786        0.726           0.257   
C1_tags               ambiguous        1.000        0.655           0.386   
                      multi_domain     0.857        0.533           0.400   
                      specialistic     0.857        0.786           0.286   
C2_dist               ambiguous        0.929        0.631           0.371   
                      multi_domain     0.857        0.527           0.400   
                      specialistic     0.857        0.786           0.286   
C3_dist_tags          ambiguous        0.929        0.631           0.371   
                      multi_domain     0.786        0.510           0.386   
                      specialistic     0.857        0.786           0.286   
C4_dist_tags_joinconv ambiguous        1.000        0.726           0.429   
                      multi_domain     0.857        0.510           0.386   
                      specialistic     0.857        0.786           0.286   
C5_dist_tags_joinall  ambiguous        0.929        0.714           0.414   
                      multi_domain     0.929        0.637           0.486   
                      specialistic     0.857        0.786           0.286   

                                    gold_mrr  
config                category                
C0_baseline           ambiguous        0.595  
                      multi_domain     0.577  
                      specialistic     0.643  
C1_tags               ambiguous        0.875  
                      multi_domain     0.633  
                      specialistic     0.679  
C2_dist               ambiguous        0.857  
                      multi_domain     0.610  
                      specialistic     0.679  
C3_dist_tags          ambiguous        0.857  
                      multi_domain     0.619  
                      specialistic     0.679  
C4_dist_tags_joinconv ambiguous        0.875  
                      multi_domain     0.643  
                      specialistic     0.679  
C5_dist_tags_joinall  ambiguous        0.857  
                      multi_domain     0.708  
                      specialistic     0.679

### 7 - Versione UTENTE MEDIO: tag con errori/mancanti e distanze parziali
- **Distanze**: parte dallo stesso piano esperto e ne tiene a caso una frazione (`KEEP`, riproducibile con `SEED`); `NOISE` > 0 aggiunge scambi forte/lieve.
- **Tag**: parte dai tag esperti e: `TAG_MISSING` dei chunk restano senza tag, `TAG_WRONG` dei rimanenti riceve un tag sbagliato, nei chunk con piu' tag resta solo il primo.
Va in un **altro sidecar** (copia del sidecar *base*: stesse predq, nessuna distanza) con un secondo handler. Il giudizio sugli scenari e' un'ipotesi di lavoro (valori arbitrari ma dichiarati).

In [21]:
KEEP, SEED, NOISE = 0.6, 7, 0.0
TAG_MISSING, TAG_WRONG, TAG_SEED = 0.30, 0.15, 7
plan_user = dp.degrade_plan(plan_expert, keep=KEEP, seed=SEED, class_noise=NOISE)
plan_user_path = QDIR / "distance_plan_colosseum_user.json"
dp.save_plan(plan_user, plan_user_path)
print("esperto:", dp.plan_stats(plan_expert)); print("utente :", dp.plan_stats(plan_user))

chunks_user, tag_log = degrade_tags(list(chunks.values()), TAGS, missing_frac=TAG_MISSING, wrong_frac=TAG_WRONG, seed=TAG_SEED)
print("tag degradati:", tag_log, "| chunk senza tag:", sum(not c["tags"] for c in chunks_user), "di", len(chunks_user))
json.dump(chunks_user, open(QDIR / "chunks_colosseum_user_tags.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

user_sidecar = sidecar_path.with_name("colosseum_benchmark_user.json")
shutil.copy(base_sidecar, user_sidecar)
load_tags_into_sidecar(chunks_user, user_sidecar).release_path()          # tag degradati (azzera e riscrive i tag; predq restano)
handler_u = PipelineHandler(sidecar_path=user_sidecar, queries_path=queries_path)
bench_u = RagasBenchmark(handler_u, predetermined_path=pred_path, normal_path=norm_path,
                         out_dir=OUT_DIR,
                         top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
print(dp.apply_plan(handler_u.sidecar_manager, plan_user))
print("tag nel sidecar utente:", {k: v["count"] for k, v in handler_u.sidecar_manager.get_global_tags().items()})

esperto: {'predq_edits': 99, 'chunk_pairs': 499, 'classes': {'MOLTO_AVVICINATI': 175, 'AVVICINATI': 380, 'MOLTO_ALLONTANATI': 19, 'ALLONTANATI': 24}}
utente : {'predq_edits': 59, 'chunk_pairs': 299, 'classes': {'MOLTO_ALLONTANATI': 12, 'AVVICINATI': 221, 'ALLONTANATI': 15, 'MOLTO_AVVICINATI': 110}}
tag degradati: {'secondary_dropped': 46, 'wrong': 12, 'missing': 39} | chunk senza tag: 39 di 120
[WARNING] Creata una seconda istanza di SidecarManager per lo stesso file (/home/jovyan/tesi_graphrag/data/processed/colosseum/colosseum_benchmark_user.json). Ogni istanza mantiene una propria cache in RAM: se non e' la stessa istanza ad essere condivisa e riutilizzata ovunque, le due cache possono disallinearsi silenziosamente. Crea una sola istanza e passala esplicitamente a tutti i componenti che leggono/scrivono il sidecar.


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


<<| Batch adiacenze query pre-determinate salvato: 59 elementi processati |>>
<<| Batch adiacenze standard salvato: 299 elementi processati |>>
{'predq_edits': 59, 'chunk_pairs': 299}
tag nel sidecar utente: {'History': 16, 'Conservation': 6, 'Architecture': 22, 'Archaeology': 12, 'Culture': 10, 'Spectacles': 15}


### PILOTA D - anche l'utente medio migliora C0? (nessun LLM, ~1 minuto)
Stesso verdetto del pilota C, con tag degradati e distanze parziali. Se l'utente medio **non** migliora C0 ma l'esperto si', e' un risultato (il sistema richiede cura), ma conviene saperlo prima.

In [22]:
### PILOTA D ###
genD = bench_u.generate("pilot_user", with_answers=False)
mD = genD[META + GOLD_COLS]
rN_u = verdetto(mD, ~mD.is_predetermined, "UTENTE MEDIO - query NORMALI")
rA_u = verdetto(mD, ~mD.is_predetermined & (mD.category == "ambiguous"), "UTENTE MEDIO - AMBIGUE normali")
print("config che migliorano C0 (utente medio, normali):", sorted(set(rN_u[rN_u.migliora].config)) or "NESSUNA")

--- UTENTE MEDIO - query NORMALI


,metric,config,n,mean_diff,ci95_low,ci95_high,win,tie,loss,migliora,peggiora
5,gold_recall,C1_tags,30,-0.053,-0.183,0.072,8,12,10,False,False
6,gold_recall,C2_dist,30,-0.061,-0.192,0.067,8,12,10,False,False
7,gold_recall,C3_dist_tags,30,-0.053,-0.183,0.072,8,12,10,False,False
8,gold_recall,C4_dist_tags_joinconv,30,-0.050,-0.181,0.072,8,12,10,False,False
9,gold_recall,C5_dist_tags_joinall,30,0.075,-0.003,0.156,9,18,3,False,False
15,gold_mrr,C1_tags,30,-0.013,-0.198,0.165,8,14,8,False,False
16,gold_mrr,C2_dist,30,-0.019,-0.206,0.164,8,14,8,False,False
17,gold_mrr,C3_dist_tags,30,-0.013,-0.198,0.165,8,14,8,False,False
18,gold_mrr,C4_dist_tags_joinconv,30,-0.002,-0.182,0.176,8,14,8,False,False
19,gold_mrr,C5_dist_tags_joinall,30,0.106,-0.022,0.239,8,19,3,False,False


--- UTENTE MEDIO - AMBIGUE normali


,metric,config,n,mean_diff,ci95_low,ci95_high,win,tie,loss,migliora,peggiora
5,gold_recall,C1_tags,10,0.100,-0.133,0.333,5,3,2,False,False
6,gold_recall,C2_dist,10,0.100,-0.133,0.333,5,3,2,False,False
7,gold_recall,C3_dist_tags,10,0.100,-0.133,0.333,5,3,2,False,False
8,gold_recall,C4_dist_tags_joinconv,10,0.133,-0.100,0.333,5,4,1,False,False
9,gold_recall,C5_dist_tags_joinall,10,0.167,0.067,0.267,5,5,0,True,False
15,gold_mrr,C1_tags,10,0.083,-0.300,0.467,4,3,3,False,False
16,gold_mrr,C2_dist,10,0.083,-0.300,0.467,4,3,3,False,False
17,gold_mrr,C3_dist_tags,10,0.083,-0.300,0.467,4,3,3,False,False
18,gold_mrr,C4_dist_tags_joinconv,10,0.103,-0.270,0.470,4,3,3,False,False
19,gold_mrr,C5_dist_tags_joinall,10,0.150,-0.150,0.450,3,5,2,False,False


config che migliorano C0 (utente medio, normali): NESSUNA


In [23]:
gen_u = bench_u.generate("post_edit_user")
print("predq non riconosciute/errate:", gen_u[~gen_u.predq_ok].query_id.unique())
display(bench_u.retrieval_summary("post_edit_user"))

predq non riconosciute/errate: <ArrowStringArray>
[]
Length: 0, dtype: str


gold_hit  gold_recall  gold_precision  \
config                category                                              
C0_baseline           ambiguous        0.714        0.345           0.200   
                      multi_domain     0.786        0.448           0.343   
                      specialistic     0.786        0.726           0.257   
C1_tags               ambiguous        0.786        0.476           0.286   
                      multi_domain     0.857        0.419           0.314   
                      specialistic     0.786        0.583           0.200   
C2_dist               ambiguous        0.786        0.476           0.286   
                      multi_domain     0.786        0.401           0.300   
                      specialistic     0.786        0.583           0.200   
C3_dist_tags          ambiguous        0.786        0.476           0.286   
                      multi_domain     0.857        0.419           0.314   
                      specialistic     0.786        0.583           0.200   
C4_dist_tags_joinconv ambiguous        0.857        0.500           0.300   
                      multi_domain     0.857        0.401           0.300   
                      specialistic     0.786        0.583           0.200   
C5_dist_tags_joinall  ambiguous        0.929        0.619           0.357   
                      multi_domain     0.929        0.531           0.400   
                      specialistic     0.929        0.798           0.286   

                                    gold_mrr  
config                category                
C0_baseline           ambiguous        0.595  
                      multi_domain     0.577  
                      specialistic     0.643  
C1_tags               ambiguous        0.667  
                      multi_domain     0.610  
                      specialistic     0.631  
C2_dist               ambiguous        0.667  
                      multi_domain     0.595  
                      specialistic     0.631  
C3_dist_tags          ambiguous        0.667  
                      multi_domain     0.610  
                      specialistic     0.631  
C4_dist_tags_joinconv ambiguous        0.681  
                      multi_domain     0.655  
                      specialistic     0.631  
C5_dist_tags_joinall  ambiguous        0.821  
                      multi_domain     0.738  
                      specialistic     0.726

Per ripetere con altri `KEEP`/`SEED`/`TAG_SEED` (es. 5 seed per una media): cambia i parametri, usa un run_id nuovo (`post_edit_user_s8`) e ripeti la cella 7 (rifa' la copia del sidecar *base*).

### 8 - RAGAS (giudice esterno) su Google Colab
I file di generazione sono in `data/benchmark/colosseum/`: **`post_edit_expert_generation.jsonl`** e **`post_edit_user_generation.jsonl`**.
1. Su Drive crea la cartella **`tesi_ragas_colosseum`** e carica i due file (non i `pilot_*`).
2. Apri `RagasColab_T4_v3.ipynb`: `DATASET = "colosseum"`, `RUN = "post_edit_expert"` (poi `post_edit_user`), `METRIC_SET = "extended"`.
3. Scarica i CSV `post_edit_*_ragas.csv` (e, se lanci `METRIC_SET = "no_reference"`, i `*_noref.csv`) in `data/benchmark/colosseum/`.
- C0 si valuta **solo** nel run `post_edit_expert` (6 config); `post_edit_user` ha 5 config. In totale 11 CSV.
- Metriche `extended` = Faithfulness, ResponseRelevancy, ContextPrecision/Recall (con risposta umana) + **FactualCorrectness** (il giudice confronta le affermazioni della risposta del sistema con la risposta umana) + SemanticSimilarity.
- `no_reference` = senza risposta umana: Faithfulness, ResponseRelevancy, ContextPrecision senza reference (richiesta del relatore: vedere se le due modalita' concordano).
- Per ridurre il tempo: `ONLY_NORMAL = True` (30 query) e/o `CONFIGS = ["C0_baseline", "C1_tags", "C3_dist_tags", "C4_dist_tags_joinconv"]`.

In [ ]:
### REPORT ### (dopo aver messo i CSV in data/benchmark/colosseum/)
exp = bench.load_ragas("post_edit_expert"); usr = bench.load_ragas("post_edit_user")
base = exp[exp.config == "C0_baseline"]       # C0 non dipende da tag/distanze: unica
VARIANTS = {"esperto (tag + distanze ottimali)": exp,
            "utente medio (tag con errori + distanze parziali)": pd.concat([base, usr], ignore_index=True)}
for name, df in VARIANTS.items():
    print("=====", name)
    display(bench.summary(df, by="category"))
    display(bench.summary(df, by="is_predetermined"))                    # in-sample (ritoccate) vs generalizzazione
    display(bench.paired_vs_baseline(df, subset=~df.is_predetermined))   # solo normali
print("--- retrieval su gold (nessun LLM)")
display(pd.concat({r: bench.retrieval_summary(r) for r in ["post_edit_user", "post_edit_expert"]}))

In [ ]:
### VERDETTO PER IL RELATORE: con reranking (C3/C4) contro senza (C0), giudice con risposta umana e senza
import glob
KEY_CFGS = ["C0_baseline", "C3_dist_tags", "C4_dist_tags_joinconv"]
METRICS_REF = bench._metric_cols(exp)      # tutte le metriche presenti nei CSV (incl. factual_correctness e semantic_similarity se calcolate)
for name, df in VARIANTS.items():
    d = df[df.config.isin(KEY_CFGS)]
    print("=====", name, "- tutte le query / solo ambigue / solo normali")
    for label, sub in [("tutte", d), ("ambigue", d[d.category == "ambiguous"]), ("normali", d[~d.is_predetermined])]:
        print(f"[{label}]"); display(sub.groupby("config")[METRICS_REF].mean().round(3))
    print("differenze appaiate contro C0 (IC95% bootstrap, query normali):")
    display(bench.paired_vs_baseline(d, subset=~d.is_predetermined))

noref_files = sorted(glob.glob(str(OUT_DIR / "post_edit_*_noref.csv")))
if noref_files:
    nr = pd.concat([pd.read_csv(f) for f in noref_files], ignore_index=True)
    print("===== SENZA risposta umana (no_reference)")
    display(nr[nr.config.isin(KEY_CFGS)].groupby(["run_id", "config"])[bench._metric_cols(nr)].mean().round(3))
else:
    print("(nessun CSV no_reference: lancia Colab con METRIC_SET = 'no_reference' se vuoi il confronto senza risposta umana)")